# 09 — Train the V3 subject-wise ensemble

Five subject-disjoint folds train compact waveform and STFT CNNs plus an SQI
ExtraTrees model. Every development record receives an out-of-fold prediction.
Calibration and quality-gate thresholds use only these OOF scores; original test
subjects are never opened by this notebook.

In [ ]:
from pathlib import Path
import os
import sys


def find_repository_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / ".git").exists() and (candidate / "apps").exists():
            return candidate
    raise RuntimeError("Run this notebook inside the pure-pulsesure repository")


REPO_ROOT = find_repository_root(Path.cwd().resolve())
RAW_DIR = Path(os.environ.get("PURE_BUT_PPG_DIR", REPO_ROOT / "data" / "raw" / "but-ppg-2.0.0")).expanduser().resolve()
V3_DATA_DIR = Path(os.environ.get("PURE_V3_DATA_DIR", REPO_ROOT / "data" / "processed" / "but-ppg-v3")).expanduser().resolve()
V3_MODEL_DIR = Path(os.environ.get("PURE_V3_MODEL_DIR", REPO_ROOT / "models" / "production" / "v3")).expanduser().resolve()

print("Raw:", RAW_DIR)
print("V3 data:", V3_DATA_DIR)
print("V3 models:", V3_MODEL_DIR)

In [ ]:
import subprocess

subprocess.run(
    [
        sys.executable,
        str(REPO_ROOT / "scripts" / "train_v3_ensemble.py"),
        "--input-dir", str(V3_DATA_DIR),
        "--model-dir", str(V3_MODEL_DIR),
        "--epochs", os.environ.get("PURE_V3_EPOCHS", "50"),
        "--patience", os.environ.get("PURE_V3_PATIENCE", "8"),
    ],
    check=True,
)

In [ ]:
import json
import pandas as pd

report = json.loads((V3_MODEL_DIR / "oof_metrics.json").read_text())
display(pd.Series(report["oof_binary"]).to_frame("OOF ensemble"))
display(pd.Series(report["oof_gate"]).to_frame("OOF quality gate"))
display(pd.DataFrame(report["branch_oof_metrics"]).T[["accuracy", "balanced_accuracy", "f1_good", "roc_auc", "false_acceptance_rate"]])